# Systematics from grid histogram counts

Loads ``syst_hists`` produced by ``configs/numucc_1p0pi/syst_histcounts.py``, sums across files,
and builds cov/corr + uncertainty-summary plots.

By default reads from **pnfs scratch**
(``/pnfs/sbnd/scratch/users/munjung/``): smoke under ``cafpyana_tmp/…`` and grid
jobs under ``cafpyana_out/``. Override with env ``NUMUCC_SYST_HIST_ROOT``.

Binning/labels come from the **frozen VariableConfig snapshot** written with the
jobs (HDF key ``var_configs`` and/or ``variable_configs.json``) — not from the
live ``variable_configs.py`` module.

## Slim products (do not double-count)

* ``slim_multisim`` — product of true multisim knobs only
* ``slim`` — slim_multisim × Gaussian throws of multisigma/morph

Prefer ``slim`` **or** the sum of per-knob multisim covs — not both.

## GENIE rate vs xsec

* **Rate** — reweighted selected reco yields
* **Xsec** — response-matrix path: $N_u = R(u)\,N_{\mathrm{gen}}^{\mathrm{CV}} + (b_u-b_{\mathrm{CV}})$


In [5]:
from __future__ import annotations

import glob
import os
from collections import defaultdict
import sys
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana') # absolute path for running on EAF

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from makedf.geniesyst import GENIE_KNOB_GROUPS
from analysis_village.numucc_1p0pi.utils import plot_heatmap
from analysis_village.numucc_1p0pi.syst_histcounts import (
    VAR_CONFIG_SNAPSHOT_JSON_NAME,
    combine_indep_knob_frac_covs,
    finalize_genie_xsec_cv,
    finalize_genie_xsec_univ,
    load_syst_hists_from_glob,
    load_var_config_snapshot_json,
    load_var_configs_from_glob,
    nbins_by_var_from_configs,
    rate_cov_from_univ_cv,
    scale_cov_by_contamination,
    unisim_cov_from_cv_and_var,
    unpack_rate_from_df,
    unpack_xsec_from_df,
)

# --- pnfs scratch defaults (override with NUMUCC_SYST_HIST_ROOT) ---
SCRATCH = os.environ.get(
    "NUMUCC_SYST_SCRATCH",
    "/pnfs/sbnd/scratch/users/munjung",
)
HIST_ROOT = os.environ.get(
    "NUMUCC_SYST_HIST_ROOT",
    os.path.join(SCRATCH, "cafpyana_tmp", "syst_histcounts_smoke", "out"),
)
# Also search grid outputs under cafpyana_out (and smoke, if HIST_ROOT differs).
HIST_SEARCH_ROOTS = [
    HIST_ROOT,
    os.path.join(SCRATCH, "cafpyana_out"),
    os.path.join(SCRATCH, "cafpyana_tmp", "syst_histcounts_smoke", "out"),
]
# de-dupe, keep order
_seen = set()
HIST_SEARCH_ROOTS = [
    r for r in HIST_SEARCH_ROOTS if not (r in _seen or _seen.add(r))
]

XSEC_UNIT = 1.0
PLOT_VAR = os.environ.get("NUMUCC_SYST_PLOT_VAR", "integrated")  # change as needed
COV_KIND = os.environ.get("NUMUCC_SYST_COV_KIND", "rate")  # "rate" or "xsec"

# Filled after discovery from job-output snapshot (HDF ``var_configs`` / JSON).
# Do not import live ``variable_configs.py`` — it can change after jobs ran.
VAR_CONFIGS: dict = {}
NBINS_BY_VAR: dict[str, int] = {}
print("SCRATCH =", SCRATCH)
print("HIST_ROOT =", HIST_ROOT)
print("HIST_SEARCH_ROOTS =", HIST_SEARCH_ROOTS)
print("PLOT_VAR =", PLOT_VAR, "COV_KIND =", COV_KIND)


ImportError: cannot import name 'VAR_CONFIG_SNAPSHOT_JSON_NAME' from 'analysis_village.numucc_1p0pi.syst_histcounts' (/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana/analysis_village/numucc_1p0pi/syst_histcounts.py)

## 1. Discover and load histcount files

Searches ``HIST_SEARCH_ROOTS`` under ``/pnfs/sbnd/scratch/users/munjung/`` by default
(``cafpyana_tmp/syst_histcounts_smoke/out`` and ``cafpyana_out``).

| tag | content |
|-----|--------|
| ``hist_mc_genie_*`` / ``hist_dirt_all`` | GENIE (+ slim); dirt may be mode=all |
| ``hist_mc_flux`` / ``hist_mc_g4`` | Flux / G4 |
| ``hist_mc_cv_nowgt`` | CV for WireMod pairing |
| ``hist_wiremod_*`` / ``hist_dent_*`` | detector unisim |
| ``hist_offbeam`` / ``hist_intime`` | cosmics unisim |


In [7]:
def glob_hist_dfs(*tags: str, roots=None) -> list[str]:
    """Match ``*.df`` under pnfs scratch search roots (default: ``HIST_SEARCH_ROOTS``)."""
    if roots is None:
        roots = HIST_SEARCH_ROOTS
    files = []
    for root in roots:
        if not root or not os.path.isdir(root):
            continue
        for tag in tags:
            pats = [
                os.path.join(root, f"*{tag}*.df"),
                os.path.join(root, "**", f"*{tag}*.df"),
            ]
            for p in pats:
                files.extend(glob.glob(p, recursive=True))
    return sorted(set(files))


# Neutrino-weighted: split MC jobs (+ optional hist_mc_all) + dirt mode=all
MC_GENIE_FILES = glob_hist_dfs("hist_mc_genie_", "hist_mc_all")
MC_FLUX_FILES = glob_hist_dfs("hist_mc_flux")
MC_G4_FILES = glob_hist_dfs("hist_mc_g4")
DIRT_ALL_FILES = glob_hist_dfs("hist_dirt_all")
NEUTRINO_FILES = sorted(
    set(MC_GENIE_FILES + MC_FLUX_FILES + MC_G4_FILES + DIRT_ALL_FILES)
)

WIREMOD_CV_FILES = glob_hist_dfs("hist_mc_cv_nowgt")
WIREMOD_SV_FILES = glob_hist_dfs("hist_wiremod_sv")
WIREMOD_XTXW_FILES = glob_hist_dfs("hist_wiremod_xtxw")
DENT_CV_FILES = glob_hist_dfs("hist_dent_cv")
DENT_VAR_FILES = glob_hist_dfs("hist_dent_var")
OFFBEAM_FILES = glob_hist_dfs("hist_offbeam")
INTIME_FILES = glob_hist_dfs("hist_intime")

for label, files in [
    ("mc_genie", MC_GENIE_FILES),
    ("mc_flux", MC_FLUX_FILES),
    ("mc_g4", MC_G4_FILES),
    ("dirt_all", DIRT_ALL_FILES),
    ("neutrino_all", NEUTRINO_FILES),
    ("wiremod_cv", WIREMOD_CV_FILES),
    ("wiremod_sv", WIREMOD_SV_FILES),
    ("wiremod_xtxw", WIREMOD_XTXW_FILES),
    ("dent_cv", DENT_CV_FILES),
    ("dent_var", DENT_VAR_FILES),
    ("offbeam", OFFBEAM_FILES),
    ("intime", INTIME_FILES),
]:
    print(f"{label:14s} {len(files):3d} files")

# Frozen VariableConfig from job outputs (HDF var_configs or variable_configs.json).
_ALL_HIST_FILES = (
    NEUTRINO_FILES
    + WIREMOD_CV_FILES
    + WIREMOD_SV_FILES
    + WIREMOD_XTXW_FILES
    + DENT_CV_FILES
    + DENT_VAR_FILES
    + OFFBEAM_FILES
    + INTIME_FILES
)
VAR_CONFIGS = load_var_configs_from_glob(_ALL_HIST_FILES)
if not VAR_CONFIGS:
    for root in HIST_SEARCH_ROOTS:
        snap = os.path.join(root, VAR_CONFIG_SNAPSHOT_JSON_NAME)
        if os.path.isfile(snap):
            VAR_CONFIGS = load_var_config_snapshot_json(snap)
            print("loaded VariableConfig snapshot:", snap)
            break
NBINS_BY_VAR = nbins_by_var_from_configs(VAR_CONFIGS)
print(
    f"VAR_CONFIGS: {len(VAR_CONFIGS)} vars"
    + (f" (e.g. {sorted(VAR_CONFIGS)[:3]}…)" if VAR_CONFIGS else " — missing; re-run hist jobs")
)


mc_genie         3 files
mc_flux          1 files
mc_g4            1 files
dirt_all         1 files
neutrino_all     6 files
wiremod_cv       1 files
wiremod_sv       1 files
wiremod_xtxw     1 files
dent_cv          1 files
dent_var         1 files
offbeam          1 files
intime           1 files


NameError: name 'load_var_configs_from_glob' is not defined

## 2. Build covariances from histcounts

In [ ]:
SLIM_SKIP = {
    "slim", "slim_multisim",
    "Flux_slim", "Flux_slim_multisim",
    "G4_slim", "G4_slim_multisim",
    "GENIE", "Flux", "G4",
}


def knobs_for_genie_mode(mode: str) -> set[str]:
    return set(GENIE_KNOB_GROUPS.get(mode, []))


def pack_frac_diag(pack: dict) -> np.ndarray:
    return np.sqrt(np.clip(np.diag(pack["cov_frac"]), 0.0, None))


def build_genie_covs_from_hist_df(hist_df: pd.DataFrame, *, xsec_unit: float = 1.0):
    """Per-knob + totals. Keys: ``{knob}_rate``, ``{knob}`` (xsec), ``genie_rate``, ``genie``."""
    rate = unpack_rate_from_df(hist_df, family="GENIE", nbins_by_var=NBINS_BY_VAR)
    xsec = unpack_xsec_from_df(hist_df, family="GENIE", nbins_by_var=NBINS_BY_VAR)

    by_var: dict = defaultdict(dict)
    rate_packs_by_var: dict = defaultdict(list)
    xsec_packs_by_var: dict = defaultdict(list)
    cv_rate_by_var: dict = {}
    cv_xsec_by_var: dict = {}

    for knob, vars_d in rate.items():
        for slug, pack in vars_d.items():
            if pack["univ"].size == 0:
                continue
            rp = rate_cov_from_univ_cv(pack["univ"], pack["cv"])
            by_var[slug][f"{knob}_rate"] = rp
            if knob not in SLIM_SKIP:
                rate_packs_by_var[slug].append(rp)
            cv_rate_by_var[slug] = pack["cv"]

    for knob, vars_d in xsec.items():
        for slug, acc in vars_d.items():
            univ = finalize_genie_xsec_univ(acc, xsec_unit=xsec_unit)
            cv = finalize_genie_xsec_cv(acc, xsec_unit=xsec_unit, bkgd_subtract=True)
            if univ.size == 0:
                continue
            xp = rate_cov_from_univ_cv(univ, cv)
            by_var[slug][knob] = xp
            if knob not in SLIM_SKIP:
                xsec_packs_by_var[slug].append(xp)
            cv_xsec_by_var[slug] = cv

    for slug in set(rate_packs_by_var) | set(xsec_packs_by_var):
        if slug in rate_packs_by_var:
            by_var[slug]["genie_rate"] = combine_indep_knob_frac_covs(
                rate_packs_by_var[slug], cv_rate_by_var[slug]
            )
        if slug in xsec_packs_by_var:
            by_var[slug]["genie"] = combine_indep_knob_frac_covs(
                xsec_packs_by_var[slug], cv_xsec_by_var[slug]
            )
    return dict(by_var)


def build_multisim_rate_covs(hist_df: pd.DataFrame, family: str):
    rate = unpack_rate_from_df(hist_df, family=family, nbins_by_var=NBINS_BY_VAR)
    by_var: dict = defaultdict(dict)
    packs_by_var: dict = defaultdict(list)
    cv_by_var: dict = {}
    for knob, vars_d in rate.items():
        for slug, pack in vars_d.items():
            if pack["univ"].size == 0:
                continue
            rp = rate_cov_from_univ_cv(pack["univ"], pack["cv"])
            by_var[slug][knob] = rp
            if knob not in SLIM_SKIP:
                packs_by_var[slug].append(rp)
            cv_by_var[slug] = pack["cv"]
    key = family.lower()
    for slug, packs in packs_by_var.items():
        by_var[slug][key] = combine_indep_knob_frac_covs(packs, cv_by_var[slug])
    return dict(by_var)


def unisim_cv_counts(hist_df: pd.DataFrame) -> dict[str, np.ndarray]:
    rate = unpack_rate_from_df(hist_df, family="UNISIM", nbins_by_var=NBINS_BY_VAR)
    out = {}
    for _knob, vars_d in rate.items():
        for slug, pack in vars_d.items():
            out[slug] = np.asarray(pack["cv"], dtype=float)
    return out


def build_unisim_covs(cv_counts: dict, var_counts: dict) -> dict:
    out = {}
    for slug in sorted(set(cv_counts) & set(var_counts)):
        out[slug] = unisim_cov_from_cv_and_var(cv_counts[slug], var_counts[slug])
    return out


def sum_mode_frac_covs(
    by_var_knob: dict, mode: str, *,
    kind: str = "rate",
    cv: np.ndarray | None = None,
) -> dict | None:
    """Sum independent knob frac-covs belonging to a GENIE mode (CCQE, MEC, …)."""
    want = knobs_for_genie_mode(mode)
    packs = []
    for key, pack in by_var_knob.items():
        if key in ("genie", "genie_rate") or key in SLIM_SKIP:
            continue
        knob = key[:-5] if key.endswith("_rate") else key
        is_rate_key = key.endswith("_rate")
        if kind == "rate":
            # Prefer explicit ``*_rate`` keys when present
            if f"{knob}_rate" in by_var_knob and not is_rate_key:
                continue
            if not is_rate_key and f"{knob}_rate" not in by_var_knob:
                pass  # rate-only families store under knob name
            elif not is_rate_key:
                continue
        if kind == "xsec" and is_rate_key:
            continue
        if knob not in want:
            continue
        packs.append(pack)
    if not packs:
        return None
    if cv is None:
        # Fall back to sqrt of absolute cov diagonal / frac (reconstruction)
        cf = np.asarray(packs[0]["cov_frac"], dtype=float)
        ca = np.asarray(packs[0]["cov"], dtype=float)
        with np.errstate(divide="ignore", invalid="ignore"):
            cv = np.sqrt(np.clip(np.diag(ca) / np.clip(np.diag(cf), 1e-30, None), 0, None))
            cv = np.nan_to_num(cv, nan=1.0)
    return combine_indep_knob_frac_covs(packs, cv)

In [ ]:
genie_cov = {}
flux_cov = {}
g4_cov = {}

if NEUTRINO_FILES:
    neu_hist = load_syst_hists_from_glob(NEUTRINO_FILES)
    print(f"neutrino hist rows: {len(neu_hist)} from {len(NEUTRINO_FILES)} files")
    genie_cov = build_genie_covs_from_hist_df(neu_hist, xsec_unit=XSEC_UNIT)
    flux_cov = build_multisim_rate_covs(neu_hist, "Flux")
    g4_cov = build_multisim_rate_covs(neu_hist, "G4")
    print("GENIE vars:", sorted(genie_cov))
    print("Flux vars:", sorted(flux_cov))
    print("G4 vars:", sorted(g4_cov))
else:
    print("[warn] no hist_mc_all / hist_dirt_all files found")

detector_cov = {}
if WIREMOD_CV_FILES and WIREMOD_SV_FILES:
    cv = unisim_cv_counts(load_syst_hists_from_glob(WIREMOD_CV_FILES))
    var = unisim_cv_counts(load_syst_hists_from_glob(WIREMOD_SV_FILES))
    detector_cov["WireModYZ"] = build_unisim_covs(cv, var)
if WIREMOD_CV_FILES and WIREMOD_XTXW_FILES:
    cv = unisim_cv_counts(load_syst_hists_from_glob(WIREMOD_CV_FILES))
    var = unisim_cv_counts(load_syst_hists_from_glob(WIREMOD_XTXW_FILES))
    detector_cov["WireModXTXW"] = build_unisim_covs(cv, var)
if DENT_CV_FILES and DENT_VAR_FILES:
    cv = unisim_cv_counts(load_syst_hists_from_glob(DENT_CV_FILES))
    var = unisim_cv_counts(load_syst_hists_from_glob(DENT_VAR_FILES))
    detector_cov["DENT"] = build_unisim_covs(cv, var)
print("detector:", {k: sorted(v) for k, v in detector_cov.items()})

cosmics_cov = {}
INTIME_GATE_SCALE = 1.0  # set from hdr gates when available
FLAT_F = 0.05  # placeholder contamination; replace with measured f(bin)
if OFFBEAM_FILES and INTIME_FILES:
    off = unisim_cv_counts(load_syst_hists_from_glob(OFFBEAM_FILES))
    it = unisim_cv_counts(load_syst_hists_from_glob(INTIME_FILES))
    it = {k: v * INTIME_GATE_SCALE for k, v in it.items()}
    tmpl = build_unisim_covs(off, it)
    for slug, pack in tmpl.items():
        f = np.full(pack["cov"].shape[0], FLAT_F, dtype=float)
        cosmics_cov[slug] = scale_cov_by_contamination(pack, f)
print("cosmics vars:", sorted(cosmics_cov))

neutrino hist rows: 40290 from 1 files
GENIE vars: ['n_trks', 'nu_score']
Flux vars: ['n_trks', 'nu_score']
G4 vars: ['n_trks', 'nu_score']


IndexError: index 32 is out of bounds for axis 0 with size 32

## 3. Covariance & correlation for a selected knob

Set ``SELECTED_KNOB`` to a CAF knob name (or ``slim`` / a WireMod tag).

In [ ]:
slug = PLOT_VAR
vc = VAR_CONFIGS.get(slug)
assert vc is not None, f"unknown PLOT_VAR={slug!r}; choose from {sorted(VAR_CONFIGS)}"

# Pick first available GENIE knob on this var (edit SELECTED_KNOB to override)
genie_keys = [
    k for k in sorted(genie_cov.get(slug, {}))
    if k not in ("genie", "genie_rate") and k not in SLIM_SKIP and not k.endswith("_rate")
]
SELECTED_KNOB = os.environ.get("NUMUCC_SYST_SELECTED_KNOB", genie_keys[0] if genie_keys else "")
print("SELECTED_KNOB =", SELECTED_KNOB)

pack = None
if slug in genie_cov:
    if COV_KIND == "rate":
        pack = genie_cov[slug].get(f"{SELECTED_KNOB}_rate") or genie_cov[slug].get(SELECTED_KNOB)
    else:
        pack = genie_cov[slug].get(SELECTED_KNOB)
if pack is None and slug in flux_cov:
    pack = flux_cov[slug].get(SELECTED_KNOB)
if pack is None and slug in g4_cov:
    pack = g4_cov[slug].get(SELECTED_KNOB)
if pack is None:
    for tag, d in detector_cov.items():
        if SELECTED_KNOB in (tag, "") and slug in d:
            pack = d[slug]
            SELECTED_KNOB = tag
            break

if pack is None:
    print("[skip] no pack for", SELECTED_KNOB, "on", slug)
else:
    cov = np.asarray(pack["cov_frac"], dtype=float)
    corr = np.asarray(pack.get("corr", pack.get("corr_frac")), dtype=float)
    if corr is None or corr.size == 0:
        d = np.sqrt(np.clip(np.diag(cov), 0, None))
        with np.errstate(divide="ignore", invalid="ignore"):
            corr = cov / np.outer(d, d)
            corr = np.nan_to_num(corr, nan=0.0)
    xlab = vc.var_labels[0] if isinstance(vc.var_labels, (list, tuple)) else str(vc.var_labels)
    plot_heatmap(
        cov, vc.bins,
        plot_labels=[xlab, xlab, f"Frac. cov ({SELECTED_KNOB})"],
        cmap="viridis",
    )
    plot_heatmap(
        corr, vc.bins,
        plot_labels=[xlab, xlab, f"Correlation ({SELECTED_KNOB})"],
        cmap="bwr",
    )

## 4. Systematic uncertainty summary — selected knobs

In [ ]:
def plot_unc_summary(series: dict[str, np.ndarray], title: str, vc):
    """``series`` maps label → fractional unc array (same nbins)."""
    if not series:
        print("[skip]", title)
        return
    centers = np.asarray(vc.bin_centers, dtype=float)
    plt.figure(figsize=(8, 4))
    for label, frac in series.items():
        plt.step(centers, 100.0 * np.asarray(frac), where="mid", label=label)
    # quadrature total if ≥2 components
    if len(series) >= 2:
        stack = np.vstack([np.asarray(v, dtype=float) ** 2 for v in series.values()])
        tot = np.sqrt(np.sum(stack, axis=0))
        plt.step(centers, 100.0 * tot, where="mid", color="k", lw=2, label="Total (quad.)")
    xlab = vc.var_labels[0] if isinstance(vc.var_labels, (list, tuple)) else str(vc.var_labels)
    plt.xlabel(xlab)
    plt.ylabel("Frac. unc. [%]")
    plt.title(title)
    plt.legend(fontsize=8, ncol=2, frameon=False)
    plt.tight_layout()
    plt.show()


# Edit this list, or leave empty to auto-pick a few GENIE knobs
SELECTED_KNOBS = []
if not SELECTED_KNOBS and slug in genie_cov:
    SELECTED_KNOBS = genie_keys[:6]

knob_series = {}
for kn in SELECTED_KNOBS:
    p = None
    if slug in genie_cov:
        p = (
            genie_cov[slug].get(f"{kn}_rate") if COV_KIND == "rate"
            else genie_cov[slug].get(kn)
        ) or genie_cov[slug].get(kn)
    if p is not None:
        short = kn.split("_")[-1] if "_" in kn else kn
        knob_series[short] = pack_frac_diag(p)

plot_unc_summary(knob_series, f"Selected knobs — {slug} ({COV_KIND})", vc)

## 5. GENIE mode summary (CCQE, MEC, …)

Sums fractional covariances of knobs belonging to each ``GENIE_KNOB_GROUPS`` mode.

In [ ]:
MODE_ORDER = ["CCQE", "ZExp", "MEC", "RES", "nonRES", "DIS", "Other", "Ar23p"]
mode_series = {}
if slug in genie_cov:
    for mode in MODE_ORDER:
        mp = sum_mode_frac_covs(genie_cov[slug], mode, kind=COV_KIND)
        if mp is not None:
            mode_series[mode] = pack_frac_diag(mp)

plot_unc_summary(mode_series, f"GENIE modes — {slug} ({COV_KIND})", vc)

## 6. All-category summary — GENIE, Flux, G4, detector, cosmics

In [ ]:
cat_series = {}

if slug in genie_cov:
    gkey = "genie_rate" if COV_KIND == "rate" else "genie"
    if gkey in genie_cov[slug]:
        cat_series["GENIE"] = pack_frac_diag(genie_cov[slug][gkey])

if slug in flux_cov and "flux" in flux_cov[slug]:
    cat_series["Flux"] = pack_frac_diag(flux_cov[slug]["flux"])
if slug in g4_cov and "g4" in g4_cov[slug]:
    cat_series["G4"] = pack_frac_diag(g4_cov[slug]["g4"])

# Detector: sum WireMod + DENT frac covs in quadrature via combine if CVs match
det_packs = []
det_cv = None
for tag, d in detector_cov.items():
    if slug in d:
        det_packs.append(d[slug])
        det_cv = d[slug].get("cv")
        cat_series[tag] = pack_frac_diag(d[slug])
if len(det_packs) >= 2 and det_cv is not None:
    cat_series["Detector (sum)"] = pack_frac_diag(
        combine_indep_knob_frac_covs(det_packs, det_cv)
    )

if slug in cosmics_cov:
    cat_series["Cosmics"] = pack_frac_diag(cosmics_cov[slug])

plot_unc_summary(cat_series, f"All categories — {slug} ({COV_KIND})", vc)

## Notes

* Defaults live on **pnfs scratch**: smoke
  ``/pnfs/sbnd/scratch/users/munjung/cafpyana_tmp/syst_histcounts_smoke/out``,
  grid jobs under ``…/cafpyana_out/``.
* Smoke used reduced universe counts; production should keep full ``NUNIV``.
* Intime/offbeam single-file smokes may have empty selected counts — cosmics band then absent.
